# O3 — Commonality Analysis (Theory-Driven): SpeechGraph vs. Symptoms → Perceived Ease (SSD)

Theory-driven pre-specified commonality analysis (6B).

**Pipeline:**
- Speech-graph metrics are read from `data_SpeechGraph2.csv` (precomputed by `visualization.ipynb`)
- Symptoms are read directly from `data_SpeechGraph2.csv`
- `Moyenne_facilite_totale` is obtained by merging with the clinical database

**Speech-graph metrics (pre-specified):** `sg_aspl`, `sg_avg_degree`, `sg_lscc_size`, `sg_lscc_ratio`, `sg_tokens`  
**Symptom block (6 dimensions):** `sx_pos`, `sx_neg`, `sx_cog`, `sx_anx`, `sx_host`, `sx_disorg`  
**Outcome variable:** `Moyenne_facilite_totale` (SSD only)

---
## Table of Contents

- [1. Speech-Graph Metric Computation](#1.-Speech-Graph-Metric-Computation)
- [2. Participant-Level Dataset (SSD only)](#2.-Participant-Level-Dataset-(SSD-only))
- [3. Commonality Analysis — 6B (Theory-Driven)](#3.-Commonality-Analysis-—-6B-(Theory-Driven))

In [ ]:
# Standard library
from pathlib import Path
import warnings

# Data & numerics
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Circle
from matplotlib.colors import to_rgba

# Machine learning & statistics
from sklearn.preprocessing import StandardScaler
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Paths
ROOT        = Path('..').resolve()
DATA_DIR    = ROOT / 'SpeechGraph'
RESUL_DIR   = ROOT / 'resultats' / 'O3_computed'
RESUL_DIR.mkdir(parents=True, exist_ok=True)

SG2_PATH    = DATA_DIR / 'data_SpeechGraph2.csv'
BANQUE_PATH = ROOT / 'data' / 'Banque_pour_Evann.csv'

# Pre-specified speech-graph metrics (ASPLND and sg_ttr excluded — not in precomputed CSV)
SG_METRICS = ['ASPL', 'n_nodes', 'n_edges', 'n_tokens', 'AD', 'LSCC', 'LSCC_ratio']
# Symptom subscales
SX_COLS    = ['sx_pos', 'sx_neg', 'sx_cog', 'sx_anx', 'sx_host', 'sx_disorg']

print(f'ROOT    : {ROOT}')
print(f'Output  : {RESUL_DIR}')
print(f'SG2 data: {SG2_PATH}')

## 1. Speech-Graph Metric Computation

- Source: `data_SpeechGraph2.csv` (precomputed by `visualization.ipynb`)
- `LSCC_ratio` computed inline as `LSCC / n_nodes`
- Merged with perceived ease scores from the clinical database

In [ ]:
def _load_banque_facilite():
    """Load perceived ease scores (Moyenne_facilite_totale) for the SZ group from the clinical database."""
    banque = pd.read_csv(BANQUE_PATH, sep=';', encoding='latin-1', low_memory=False)
    banque.columns = banque.columns.str.strip()
    fac_col = [c for c in banque.columns if 'facilit' in c.lower() and 'tot' in c.lower()][0]
    bfac = banque[banque['Groupe'] == 'SZ'][['No-projet1-sujet', fac_col]].copy()
    bfac = bfac.rename(columns={'No-projet1-sujet': 'No', fac_col: 'Moyenne_facilite_totale'})
    bfac['No'] = bfac['No'].astype(str).str.strip()
    return bfac

# Load precomputed metrics from visualization.ipynb output
df_raw = pd.read_csv(SG2_PATH, sep=';')
df_raw.columns = df_raw.columns.str.strip()
df_raw = df_raw.rename(columns={'No anonyme': 'No'})
df_raw['No'] = df_raw['No'].astype(str).str.strip()

# Compute LSCC_ratio from precomputed columns
df_raw['LSCC_ratio'] = df_raw['LSCC'] / df_raw['n_nodes'].replace(0, float('nan'))

# Merge with perceived ease scores
df_raw = df_raw.merge(_load_banque_facilite(), on='No', how='left')

sg_available = [c for c in SG_METRICS if c in df_raw.columns]
print(f'Shape                  : {df_raw.shape}')
print(f'SG columns             : {sg_available}')
print(f'Moyenne_facilite_totale: {df_raw["Moyenne_facilite_totale"].notna().sum()} non-null')
if 'Groupe' in df_raw.columns:
    print(f'Groups                 : {df_raw["Groupe"].value_counts().to_dict()}')

## 2. Participant-Level Dataset (SSD only)

- Filter `Groupe == 100` (SSD)
- Aggregation per participant (mean across stories)
- Merge with clinical database for `Moyenne_facilite_totale`

In [ ]:
# SSD only, aggregate per participant
df_ssd_raw = df_raw[df_raw['Groupe'] == 100].copy()

agg_cols = [c for c in SG_METRICS + SX_COLS + ['Moyenne_facilite_totale'] if c in df_ssd_raw.columns]
agg_df = (
    df_ssd_raw
    .groupby('No')[agg_cols]
    .mean()
    .reset_index()
)

required = [c for c in SG_METRICS + SX_COLS + ['Moyenne_facilite_totale'] if c in agg_df.columns]
df_ssd = agg_df.dropna(subset=required).copy()
n_ssd = len(df_ssd)

print(f'SSD participants with complete data: {n_ssd}')
display(df_ssd[['No', 'Moyenne_facilite_totale'] + SX_COLS[:2] + SG_METRICS[:3]].head())

In [ ]:
# Standardize all variables
for col in ['Moyenne_facilite_totale'] + SX_COLS + SG_METRICS:
    if col in df_ssd.columns:
        df_ssd[f'{col}_z'] = StandardScaler().fit_transform(df_ssd[[col]])

df_ssd['facilite_z'] = df_ssd['Moyenne_facilite_totale_z']

SX_TERMS_Z = [f'{c}_z' for c in SX_COLS if f'{c}_z' in df_ssd.columns]
SG_TERMS_Z = [f'{m}_z' for m in SG_METRICS if f'{m}_z' in df_ssd.columns]

print(f'Symptom terms : {SX_TERMS_Z}')
print(f'SG terms      : {SG_TERMS_Z}')

## 3. Commonality Analysis — 6B (Theory-Driven)

Pre-specified metrics based on prior hypotheses: `sg_aspl`, `sg_avg_degree`, `sg_lscc_size`, `sg_lscc_ratio`, `sg_tokens`.
No data-driven selection within this sample.

**Legend:**
- `A`: variance unique to symptoms
- `B`: variance unique to the SpeechGraph block
- `AB`: shared variance
- `U`: unexplained variance

In [ ]:
metric_labels = {
    'ASPL':          'ASPL (global)',
    'sg_avg_degree': 'Average degree (global)',
    'sg_lscc_size':  'LSCC size (global)',
    'sg_lscc_ratio': 'LSCC ratio (global)',
    'sg_tokens':     'N tokens (global)',
    'ASPLND':        'ASPL (ND)',
}


def run_commonality(df, top_m, top_mz, label, output_stem,
                    color_a='#E63946', color_b='#FF8C42',
                    symptom_terms=None,
                    symptom_label='Symptoms\n(6 dims)',
                    sg_label='SpeechGraph\n(theory-driven)'):
    """Run a two-set commonality analysis (Symptoms vs. SpeechGraph → perceived ease).

    Fits three OLS models (symptoms-only, SpeechGraph-only, full) and decomposes
    R² into unique (A, B) and shared (AB) components. Produces a Venn diagram and
    a partial-regression plot.

    Parameters
    ----------
    df            : participant-level DataFrame with z-scored columns
    top_m         : list of raw metric names (for display)
    top_mz        : list of z-scored metric column names used in the formula
    label         : string label for printed output
    output_stem   : filename stem (unused — figures displayed only)
    color_a/b     : hex colours for the symptoms and SpeechGraph circles
    symptom_terms : override for the symptom z-score columns (default: SX_TERMS_Z)
    symptom_label / sg_label : circle labels in the Venn diagram

    Returns
    -------
    dict with keys: label, metrics, r2_sx, r2_sg, r2_full, A, B, AB, U,
                    f_change, p_change, cohen_f2, sr2
    """
    if symptom_terms is None:
        symptom_terms = SX_TERMS_Z

    sg_terms = ' + '.join(top_mz)
    sx_terms = ' + '.join(symptom_terms)

    m_sx   = smf.ols(f'facilite_z ~ {sx_terms}', data=df).fit()
    m_sg   = smf.ols(f'facilite_z ~ {sg_terms}', data=df).fit()
    m_full = smf.ols(f'facilite_z ~ {sx_terms} + {sg_terms}', data=df).fit()

    r2_sx   = m_sx.rsquared
    r2_sg   = m_sg.rsquared
    r2_full = m_full.rsquared
    A  = r2_full - r2_sg
    B  = r2_full - r2_sx
    AB = r2_sx + r2_sg - r2_full
    U  = 1 - r2_full

    f_ch, p_ch, df_diff = m_full.compare_f_test(m_sx)
    df_resid = int(m_full.df_resid)
    f2 = B / max(1 - r2_full, 1e-12)

    df['_sg_block'] = sum(m_full.params.get(mz, 0) * df[mz] for mz in top_mz)
    resid_sg   = smf.ols(f'_sg_block ~ {sx_terms}', data=df).fit().resid
    resid_ease = m_sx.resid
    sr_r, sr_p = stats.pearsonr(resid_sg, resid_ease)
    sr2 = sr_r ** 2

    print(f'\n{"=" * 70}')
    print(f'Commonality — {label}  (n = {len(df)} SSD)')
    print(f'{"=" * 70}')
    print(f'Metrics          : {top_m}')
    print(f'R²(symptoms)     = {r2_sx:.4f}')
    print(f'R²(speech-only)  = {r2_sg:.4f}')
    print(f'R²(full)         = {r2_full:.4f}')
    print(f'A (unique sx)    = {A:.3f}  ({A*100:.1f}%)')
    print(f'B (unique SG)    = {B:.3f}  ({B*100:.1f}%)')
    print(f'AB (shared)      = {AB:.3f}  ({AB*100:.1f}%)')
    print(f'U (unexplained)  = {U:.3f}  ({U*100:.1f}%)')
    p_str = f'{p_ch:.4f}' if p_ch >= 0.001 else '< 0.001'
    print(f'F-change: F({int(df_diff)},{df_resid}) = {f_ch:.3f}, p = {p_str}')
    print(f'Cohen f²(SG)     = {f2:.3f}')
    print(f'partial R² (SG)  = {sr2:.3f}, p = {sr_p:.4f}')

    mpl.rcParams['font.family'] = 'Times New Roman'

    # ── Venn diagram ──────────────────────────────────────────────────
    CX_SX, CX_SG, CY, R = -1.1, 1.1, 0.2, 1.62
    fig, ax = plt.subplots(figsize=(8.0, 5.2))
    ax.set_xlim(-3.2, 3.2)
    ax.set_ylim(-2.5, 2.9)
    ax.set_aspect('equal')
    ax.axis('off')
    fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)

    ax.add_patch(Circle((CX_SX, CY), R, facecolor=to_rgba(color_a, 0.27), edgecolor='black', lw=1.4))
    ax.add_patch(Circle((CX_SG, CY), R, facecolor=to_rgba(color_b, 0.27), edgecolor='black', lw=1.4))

    ax.text(CX_SX - 0.75, CY, f'A\n{A:.3f}\n({A*100:.1f}%)',
            ha='center', va='center', fontsize=16, fontweight='bold', color=color_a)
    ab_txt = (f'AB\n{AB:.3f}\n({AB*100:.1f}%)' if AB >= 0
              else f'AB\n{AB:.3f}\n(suppression)')
    ax.text(0, CY, ab_txt, ha='center', va='center', fontsize=17, fontweight='bold', color='#333')
    ax.text(CX_SG + 0.75, CY, f'B\n{B:.3f}\n({B*100:.1f}%)',
            ha='center', va='center', fontsize=16, fontweight='bold', color=color_b)

    ax.text(CX_SX - 0.45, CY + R + 0.10, symptom_label,
            ha='center', va='bottom', fontsize=16, color=color_a, fontweight='bold')
    ax.text(CX_SG + 0.45, CY + R + 0.10, sg_label,
            ha='center', va='bottom', fontsize=16, color=color_b, fontweight='bold')
    ax.text(CX_SX - 0.45, CY - R - 0.16, f'R²(symptoms) = {r2_sx:.3f}',
            ha='center', va='top', fontsize=15, color=color_a)
    ax.text(CX_SG + 0.45, CY - R - 0.16, f'R²(speech-only) = {r2_sg:.3f}',
            ha='center', va='top', fontsize=15, color=color_b)
    ax.text(0, CY - R - 0.65, f'Unexplained (U) = {U:.3f}  ({U*100:.1f}%)',
            ha='center', va='top', fontsize=15, color='#555')

    plt.show()

    # ── Partial regression plot ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6.8, 5.4))
    ax.scatter(resid_sg, resid_ease, alpha=0.8, color=color_a, edgecolor='white', linewidth=0.6)
    if np.std(resid_sg) > 0:
        slope, intercept = np.polyfit(resid_sg, resid_ease, 1)
        xs = np.linspace(resid_sg.min(), resid_sg.max(), 200)
        ax.plot(xs, slope * xs + intercept, color='black', lw=2)
    p_txt = f'{sr_p:.3f}' if sr_p >= 0.001 else '<0.001'
    ax.text(0.02, 0.98, f'partial R² = {sr2:.3f}\np = {p_txt}',
            transform=ax.transAxes, va='top', ha='left', fontsize=20,
            bbox=dict(boxstyle='square,pad=0.4', facecolor='white', edgecolor='#aaa', alpha=0.95))
    ax.set_xlabel('Residualized SpeechGraph block score\n(controlling symptoms)', fontsize=20)
    ax.set_ylabel('Residualized perceived ease\n(controlling symptoms)', fontsize=20)
    ax.tick_params(labelsize=16)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(alpha=0.15, linestyle='--')
    plt.tight_layout()
    plt.show()

    return dict(
        label=label, metrics='; '.join(top_m),
        r2_sx=r2_sx, r2_sg=r2_sg, r2_full=r2_full,
        A=A, B=B, AB=AB, U=U,
        f_change=f_ch, p_change=p_ch, cohen_f2=f2, sr2=sr2,
    )

In [ ]:
res_th = run_commonality(
    df_ssd.copy(),
    top_m=SG_METRICS,
    top_mz=SG_TERMS_Z,
    label='Theory-driven (ASPL + AD + LSCC size + LSCC ratio + Tokens)',
    output_stem='facilite_6B_theory',
    color_a='#E63946',
    color_b='#FF8C42',
    symptom_label='Symptoms\n(6 dims)',
    sg_label='SpeechGraph\n(theory-driven)',
)